# Mixed layer heat budget analysis of ACCESS-OM2 runs - hourly evolution

This notebook contains code to produce Fig. 3 in Holmes and Malan 2026 - the hourly analysis figure.

In [ ]:
#Load required packages
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import cftime
import string
from tqdm import tqdm

import cmocean as cm
import sys, os
import datetime

from dask.distributed import Client

In [ ]:
# Load workers:
client = Client(n_workers=2)
client

In [ ]:
# change directory to Figures/ subfolder for saving images
os.chdir('access-om2-analysis/access-om2-sst-budget/Figures')

# Load data

### Define paths, region and time period to load data for

In [ ]:
base = '/scratch/e14/rmh561/access-om2/archive/025deg_jra55_iaf_cycle6_online_mlt/'  # Location of simulation output

# Choose year to consider:
output = 366 # 365 = 2018
base2 = base + 'output%03d/ocean/' % output

#reg = [135-360,175-360, -60, -20] # SE Aus (Kajtar et al. 2022)
reg = [None,None,None,None]
times = slice(None,None)
times_snap = slice(None,None) # Note; this must be 1 more than times.

# Chunks to use:
chunks2D = {'time':1,'yt_ocean':216,'xt_ocean':240}
chunks3D = {'time':1,'st_ocean':25,'yt_ocean':324,'xt_ocean':360}

# Set constants:
rho0 = 1035.
Cp = 3992.10322329649

### Load standard daily and grid data

In [ ]:
# Grid file:
ds_grid = xr.open_dataset(base2 + 'ocean_grid.nc',chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))#.isel(time=times)

# Standard daily variables:
ds_day = xr.open_dataset(base2 + 'ocean_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day = ds_day.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day.time.values]})
ds_day.average_DT.data = ds_day.average_DT*np.timedelta64(1,'D')
ds_day = ds_day.sel(time=times)

# Standard hourly variables:
ds_hour = xr.open_dataset(base2 + 'ocean_hourly.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_hour = ds_hour.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_hour.time.values]})
ds_hour.average_DT.data = ds_hour.average_DT*np.timedelta64(1,'D')
ds_hour = ds_hour.sel(time=times)
ds_hour_budget_3d = xr.open_dataset(base2 + 'ocean_budget_hourly_3d.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_hour_budget_3d = ds_hour_budget_3d.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_hour_budget_3d.time.values]})
ds_hour_budget_3d.average_DT.data = ds_hour_budget_3d.average_DT*np.timedelta64(1,'D')
ds_hour_budget_3d = ds_hour_budget_3d.sel(time=times)
ds_hour_budget = xr.open_dataset(base2 + 'ocean_budget_hourly.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_hour_budget = ds_hour_budget.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_hour_budget.time.values]})
ds_hour_budget.average_DT.data = ds_hour_budget.average_DT*np.timedelta64(1,'D')
ds_hour_budget = ds_hour_budget.sel(time=times)

# time step resolution variables:
ds_ts = xr.open_dataset(base2 + 'ocean_timestep.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_ts = ds_ts.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_ts.time.values]})
ds_ts.average_DT.data = ds_ts.average_DT*np.timedelta64(1,'D')

# Snapshot variables:
ds_hour_snap = xr.open_dataset(base2 + 'ocean_snapshot_hourly.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
# Add previous output for last element:
ds_day_snap_m1 = xr.open_dataset(base2.replace(str(output),str(output-1)) + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_hour_snap = xr.concat([ds_day_snap_m1.isel(time=-1),ds_hour_snap],dim='time')
ds_hour_snap = ds_hour_snap.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_hour_snap.time.values]})
ds_hour_snap = ds_hour_snap.sel(time=times)

# Daily offline analysis:
ds_day_bud_offline = xr.open_dataset('/g/data/e14/rmh561/access-om2/archive/025deg_jra55_iaf_cycle6_online_mlt/post_processed_diags/mlt_budget_offline/mlt_budget_stavg_daily_offline_output366_month01.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day_bud_offline = ds_day_bud_offline.assign_coords({'time':[np.datetime64('2019-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_bud_offline.time.values]})
ds_day_bud_offline = ds_day_bud_offline.isel(time=slice(0,7))

In [ ]:
# Load data at a single point:
lon = -200.
lat = -30.
ds_hr = ds_hour.sel(xt_ocean=lon,yt_ocean=lat,method='nearest').load()
ds_t = ds_ts.sel(xt_ocean=lon,yt_ocean=lat,method='nearest').load()
ds_hr_bud = ds_hour_budget_3d.sel(xt_ocean=lon,yt_ocean=lat,method='nearest').load()
ds_hr_bud_online = ds_hour_budget.sel(xt_ocean=lon,yt_ocean=lat,method='nearest').load()
ds_hr_snap = ds_hour_snap.sel(xt_ocean=lon,yt_ocean=lat,method='nearest').load()
ds_day_bud_offline = ds_day_bud_offline.sel(xt_ocean=lon,yt_ocean=lat,method='nearest').load()

In [ ]:
# Define functions to compute grouped budget terms (hourly analysis):
bud_tendency = 'temp_tendency_in_mld_cor'
bud_var_grps = {'advection':['temp_advection_in_mld_cor',
                             'temp_submeso_in_mld',
                             'neutral_diffusion_in_mld_temp',
                             'neutral_gm_in_mld_temp',
                             'temp_vdiffuse_k33_in_mld'],
                'vert_mixing':['temp_nonlocal_KPP_in_mld',
                               'temp_vdiffuse_diff_cbt_in_mld'],
                'surface_flux':['temp_rivermix_in_mld',
                                'temp_vdiffuse_sbc_in_mld', 
                                'frazil_3d_in_mld',
                                'sfc_hflux_pme_in_mld_cor',
                                'temp_eta_smooth_in_mld_cor'], 
                'sw_pen':['sw_heat_in_mld']}
bud_var_extras = {'shortwave':['swflx_in_mld'],
                  'longwave':['lw_heat_in_mld'],
                  'sensible':['sens_heat_in_mld'],
                  'latent':['evap_heat_in_mld']}

def compute_corrections(ds_day_budget):

    # P-E+R correction: 
    pme_cor = -ds_day_budget['pme_river_times_temp_in_mld']/rho0
    ds_day_budget['sfc_hflux_pme_in_mld_cor'] = ds_day_budget['sfc_hflux_pme_in_mld'] + pme_cor*rho0*Cp

    # Eta-smoother correction:
    eta_smoother_cor = -ds_day_budget['eta_smoother_times_temp_in_mld']/rho0
    ds_day_budget['temp_eta_smooth_in_mld_cor'] = ds_day_budget['temp_eta_smooth_in_mld'] + eta_smoother_cor*rho0*Cp

    # Advection correction:
    adv_cor1 = (-ds_day_budget['eta_t_tendency_times_temp_in_mld'] + ds_day_budget['pme_river_times_temp_in_mld'] + ds_day_budget['eta_smoother_times_temp_in_mld'])/rho0
    adv_cor2 = ds_day_budget['s_surf_ent_temp']/rho0
    ds_day_budget['temp_advection_in_mld_cor'] = ds_day_budget['temp_advection_in_mld'] + adv_cor1*rho0*Cp  + adv_cor2*rho0*Cp

    # Total correction for entrainment-by-residual:
    ds_day_budget['temp_tendency_in_mld_cor'] = ds_day_budget['temp_tendency_in_mld']  + pme_cor*rho0*Cp + adv_cor1*rho0*Cp + adv_cor2*rho0*Cp + eta_smoother_cor*rho0*Cp

    return(ds_day_budget)

def mlt_budget_fixedh(ds_day_budget, do_extras = True):

    # Extract variable sums in groups, dividing by rho0*Cp to convert to degC/second
    mlt_budget = (ds_day_budget[bud_tendency]/rho0/Cp).rename('fixedh_tendency').to_dataset()
    for var in bud_var_grps.keys():
        mlt_budget[var] = ds_day_budget[bud_var_grps[var][0]]/rho0/Cp
        if (len(bud_var_grps[var])>1):
            for raw_var in bud_var_grps[var][1:]:
                mlt_budget[var] += ds_day_budget[raw_var]/rho0/Cp
    
    # Compute residual for check:
    mlt_budget['residual'] = mlt_budget['fixedh_tendency'].copy(deep=True)
    for var in list(mlt_budget.data_vars):
        mlt_budget['residual'] -= mlt_budget[var]

    if do_extras:
        for var in bud_var_extras.keys():
            mlt_budget[var] = ds_day_budget[bud_var_extras[var][0]]/rho0/Cp
            if (len(bud_var_extras[var])>1):
                for raw_var in bud_var_extras[var][1:]:
                    mlt_budget[var] += ds_day_budget[raw_var]/rho0/Cp

    return(mlt_budget)

def compute_tendency_entrainment(mlt_budget,mlt_snap):

    mlt_snap = mlt_snap.transpose(*mlt_budget['fixedh_tendency'].dims)

    # Compute mlt tendency by taking time derivative of snapshot mlt:
    mlt_budget['mlt_tendency'] = xr.zeros_like(mlt_budget.fixedh_tendency).copy(deep=True)
    mlt_budget['mlt_tendency'].data = mlt_snap.isel(time=slice(1,None)).values - mlt_snap.isel(time=slice(0,-1)).values
    DT = xr.zeros_like(mlt_budget['mlt_tendency'].time)
    DT.data = (mlt_snap.time.isel(time=slice(1,None)).values-mlt_snap.time.isel(time=slice(0,-1)).values)/np.timedelta64(1,'s')
    mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/DT

    # Compute entrainment by residual:
    mlt_budget['entrainment'] = -(mlt_budget['fixedh_tendency'] - mlt_budget['mlt_tendency'])

    return(mlt_budget)

In [ ]:
# Compute budget terms (online):
ds_hr_bud_online = compute_corrections(ds_hr_bud_online)
ds_hr_bud_gr = mlt_budget_fixedh(ds_hr_bud_online)
ds_hr_bud_gr = compute_tendency_entrainment(ds_hr_bud_gr,ds_hr_snap.temp_in_mld/rho0)

In [ ]:
# Compute entrainment and mlt_tendency terms for offline:
ds_day_bud_offline['mlt_tendency'] = ds_hr_bud_gr.mlt_tendency.resample(time='1D').mean()
ds_day_bud_offline['entrainment'] = ds_day_bud_offline['mlt_tendency'] - ds_day_bud_offline['fixedh_tendency']

In [ ]:
fig,axes = plt.subplots(nrows=7,ncols=1,figsize=(9,14),layout='constrained',sharex=True,height_ratios=[.8,.7,1.1,1.1,1.1,1.5,1.2])

# Surface heat fluxes:
#ds_hr.swflx.plot(ax=axes[0],label='Shortwave')
ds_t.swflx.plot(ax=axes[0],label='Shortwave')
ds_hr.lw_heat.plot(ax=axes[0],label='Longwave')
ds_hr.evap_heat.plot(ax=axes[0],label='Latent')
ds_hr.sens_heat.plot(ax=axes[0],label='Sensible')
#(ds_hr.swflx+ds_hr.lw_heat+ds_hr.evap_heat+ds_hr.sens_heat).plot(ax=axes[0],label='Net')
ds_hr.swflx.resample(time='1D').mean().plot.step(ax=axes[0],where='post',color='C0',linestyle='dashed')
#(ds_hr.swflx+ds_hr.lw_heat+ds_hr.evap_heat+ds_hr.sens_heat).resample(time='1D').mean().plot.step(ax=axes[0],where='post',color='C4',linestyle='dashed')
axes[0].set_title('(a) Surface heat fluxes')
axes[0].legend()
axes[0].set_xlabel('')
axes[0].set_ylabel('Wm$^{-2}$')

# Mixed layer temperature:
ii = 1
(ds_hr_snap.temp_in_mld/rho0).plot(ax=axes[ii],label='Snapshots')
(ds_hr.temp_in_mld/rho0).plot(ax=axes[ii],label='Hourly average')
(ds_hr.temp_in_mld/rho0).resample(time='1D').mean().plot.step(ax=axes[ii],where='post',color='C1',linestyle='dashed')
axes[ii].set_title('(b) Mixed layer temperature')
axes[ii].legend()
axes[ii].set_xlabel('')
axes[ii].set_ylabel('$^\circ$C')

# Budget terms:
ii = 5
(ds_hr_bud_gr['mlt_tendency']*86400).plot(ax=axes[ii],label='Tendency',color='k',linewidth=2)
((ds_hr_bud_gr['surface_flux']+ds_hr_bud_gr['sw_pen'])*86400).plot(ax=axes[ii],label='Surface fluxes')
(ds_hr_bud_gr['advection']*86400).plot(ax=axes[ii],label='Advection')
(ds_hr_bud_gr['vert_mixing']*86400).plot(ax=axes[ii],label='Vertical mixing')
(ds_hr_bud_gr['entrainment']*86400).plot(ax=axes[ii],label='Entrainment')
#(ds_hr_snap.mld.diff('time')/10).plot(ax=axes[ii],color='k',linewidth=1.,label='dMLD/dt')
#((ds_hr_bud_gr['surface_flux']+ds_hr_bud_gr['sw_pen'])*86400).resample(time='1D').mean().plot.step(where='post',color='C0',linestyle='dashed',ax=axes[ii])
#(ds_hr_bud_gr['vert_mixing']*86400).resample(time='1D').mean().plot.step(where='post',color='C1',linestyle='dashed',ax=axes[ii])
#(ds_hr_bud_gr['entrainment']*86400).resample(time='1D').mean().plot.step(where='post',color='C2',linestyle='dashed',ax=axes[ii])
#(ds_hr_bud_gr['mlt_tendency']*86400).resample(time='1D').mean().plot.step(where='post',color='C3',linestyle='dashed',ax=axes[ii])
#(ds_hr_bud_gr['advection']*86400).resample(time='1D').mean().plot.step(where='post',color='C4',linestyle='dashed',ax=axes[ii])
axes[ii].axhline([0],color='k',linewidth=1.)
axes[ii].set_title('(f) Mixed layer temperature budget')
axes[ii].legend(loc='upper right',bbox_to_anchor=[.81,1])
axes[ii].set_xlabel('')
axes[ii].set_ylabel('$^\circ$C/day')

# Daily averages:
ii = 6
(ds_hr_bud_gr['mlt_tendency']*86400).resample(time='1D').mean().plot.step(where='post',color='k',ax=axes[ii])
((ds_hr_bud_gr['surface_flux']+ds_hr_bud_gr['sw_pen'])*86400).resample(time='1D').mean().plot.step(where='post',color='C0',ax=axes[ii],label='Online')
(ds_hr_bud_gr['advection']*86400).resample(time='1D').mean().plot.step(where='post',color='C1',ax=axes[ii])
(ds_hr_bud_gr['vert_mixing']*86400).resample(time='1D').mean().plot.step(where='post',color='C2',ax=axes[ii])
(ds_hr_bud_gr['entrainment']*86400).resample(time='1D').mean().plot.step(where='post',color='C3',ax=axes[ii])
(ds_day_bud_offline['mlt_tendency']*86400).resample(time='1D').mean().plot.step(where='post',color='k',ax=axes[ii],linestyle='dashed')
((ds_day_bud_offline['surface_flux']+ds_day_bud_offline['sw_pen'])*86400).resample(time='1D').mean().plot.step(where='post',color='C0',ax=axes[ii],linestyle='dashed',label='Daily offline')
(ds_day_bud_offline['advection']*86400).resample(time='1D').mean().plot.step(where='post',color='C1',ax=axes[ii],linestyle='dashed')
(ds_day_bud_offline['vert_mixing']*86400).resample(time='1D').mean().plot.step(where='post',color='C2',ax=axes[ii],linestyle='dashed')
(ds_day_bud_offline['entrainment']*86400).resample(time='1D').mean().plot.step(where='post',color='C3',ax=axes[ii],linestyle='dashed')
axes[ii].set_title('(g) Daily-average mixed layer temperature budget')
axes[ii].legend()
axes[ii].set_xlabel('')
axes[ii].set_ylabel('$^\circ$C/day')

# Temperature and MLD:
ii = 2
(ds_hr.temp-273.15).plot(ax=axes[ii],x='time',vmin=22,vmax=24,cmap='RdBu_r',extend='both',cbar_kwargs={'label':'$^\circ$C'})
(ds_hr.temp-273.15).plot.contour(levels=np.arange(20.,30.2,0.2),ax=axes[ii],x='time',colors='k',linewidths=0.5)
ds_hr.mld.plot(ax=axes[ii],color='k',linewidth=2.)
ds_hr.mld.resample(time='1D').mean().plot.step(ax=axes[ii],where='post', color='k',linewidth=1.,linestyle='dashed')
#ds_hr_snap.mld.plot(ax=axes[ii],color='b',linewidth=1.)
axes[ii].set_title('(c) Temperature')

# Surface heating:
ii = 3
((ds_hr_bud.sw_heat+ds_hr_bud.temp_vdiffuse_sbc)/ds_hr_bud.dzt/rho0/Cp*86400).plot(ax=axes[ii],x='time',cmap='RdBu_r',extend='both',vmin=-1.5,vmax=1.5,cbar_kwargs={'label':'$^\circ$C/day'})
ds_hr.mld.plot(ax=axes[ii],color='k',linewidth=2.)
ds_hr.mld.resample(time='1D').mean().plot.step(ax=axes[ii],where='post', color='k',linewidth=1.,linestyle='dashed')
axes[ii].set_title('(d) Surface flux-driven heating')

# Vertical mixing:
ii = 4
(ds_hr_bud.temp_vdiffuse_diff_cbt/ds_hr_bud.dzt/rho0/Cp*86400).plot(ax=axes[ii],x='time',cmap='RdBu_r',extend='both',vmin=-1.5,vmax=1.5,cbar_kwargs={'label':'$^\circ$C/day'})
ds_hr.diff_cbt_t.plot.contour(levels=np.arange(0.,0.1025,0.0025),ax=axes[ii],x='time',colors='k',linewidths=0.5)
ds_hr.mld.plot(ax=axes[ii],color='k',linewidth=2.)
ds_hr.mld.resample(time='1D').mean().plot.step(ax=axes[ii],where='post', color='k',linewidth=1.,linestyle='dashed')
axes[ii].set_title('(e) Vertical mixing-driven heating')
for ax in [axes[i] for i in [2,3,4]]:
    ax.set_xlabel('')
    ax.set_ylabel('Depth (m)')
    ax.set_ylim([0.,30.])
    ax.invert_yaxis()
    ax.set_xlim([np.datetime64('2019-01-01'),np.datetime64('2019-01-07')])
for ax in axes:
    ax.grid()
plt.savefig('Example_daily_evolution.png',dpi=200,bbox_inches='tight')

In [ ]:
fig, axes = plt.subplots(nrows=3,ncols=2,figsize=(12,12))
ds_hour.mld.sel(time='2019-01-07T04:30:00').sel(xt_ocean=slice(-220,-180),yt_ocean=slice(-50,-20)).plot(vmin=0.,vmax=80.,ax=axes[0][0])
ds_hour.mld.sel(time='2019-01-07T16:30:00').sel(xt_ocean=slice(-220,-180),yt_ocean=slice(-50,-20)).plot(vmin=0.,vmax=80.,ax=axes[1][0])
(ds_hour.mld.sel(time='2019-01-07T04:30:00')-ds_hour.mld.sel(time='2019-01-07T16:30:00')).sel(xt_ocean=slice(-220,-180),yt_ocean=slice(-50,-20)).plot(vmin=-10.,vmax=10.,cmap='RdBu_r',ax=axes[2][0])
(ds_hour.temp_in_mld.sel(time='2019-01-07T04:30:00')/rho0).sel(xt_ocean=slice(-220,-180),yt_ocean=slice(-50,-20)).plot(vmin=15,vmax=25,ax=axes[0][1])
(ds_hour.temp_in_mld.sel(time='2019-01-07T16:30:00')/rho0).sel(xt_ocean=slice(-220,-180),yt_ocean=slice(-50,-20)).plot(vmin=15,vmax=25,ax=axes[1][1])
((ds_hour.temp_in_mld.sel(time='2019-01-07T04:30:00')-ds_hour.temp_in_mld.sel(time='2019-01-07T16:30:00'))/rho0).sel(xt_ocean=slice(-220,-180),yt_ocean=slice(-50,-20)).plot(vmin=-1.,vmax=1.,cmap='RdBu_r',ax=axes[2][1])
for ax in axes.reshape(-1):
    ax.scatter([-200],[-30],s=50,color='k',marker='x')

In [ ]:
# Scaling calculation:
Hday = 5
Hnight = 20
Qday = 800.0
Qnight = 0.0
efold = 30.0
rho0 = 1025.
Cp = 4000.

SWday = Qday*(1-np.exp(-Hday/efold))
print(f'SW absorbed during day = {SWday:.1f} Wm-2')
SWnight = Qnight*(1-np.exp(-Hnight/efold))
print(f'SW absorbed during night = {SWnight:.1f} Wm-2')
print(f'Total SW absorbed = {(SWday+SWnight)/2:.1f} Wm-2')
SWavg = (Qday+Qnight)/2*(1-np.exp(-(Hday+Hnight)/2./efold))
print(f'SW absorbed for daily average = {SWavg:.1f}')

rate_day = SWday/Hday/rho0/Cp*86400.
print(f'Heating rate during day = {rate_day:.3f} degC/day')
rate_night = SWnight/Hday/rho0/Cp*86400.
print(f'Heating rate during night = {rate_night:.3f} degC/day')
print(f'Heating rate over the whole day = {(rate_day+rate_night)/2.:.3f} degC/day')
rate_avg = SWavg/(Hday+Hnight)*2/rho0/Cp*86400.
print(f'Heating rate for daily average = {rate_avg:.3f} degC/day')
print(f'Heating rate error = {((rate_day+rate_night)/2./rate_avg - 1)*100:.0f} %')


## Demonstrate shortwave radiation issue in libaccessom2

In [ ]:
# Load hourly data from ACCESS-OM2:
ds_hour = xr.open_dataset(base2 + 'ocean_hourly.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_hour = ds_hour.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_hour.time.values]})
ds_hour.average_DT.data = ds_hour.average_DT*np.timedelta64(1,'D')

# Load time step data from ACCESS-OM2:
ds_ts = xr.open_dataset(base2 + 'ocean_timestep.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_ts = ds_ts.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_ts.time.values]})
ds_ts.average_DT.data = ds_ts.average_DT*np.timedelta64(1,'D')

# Load 3-hourly data from JRA55:
ds_jra55 = xr.open_dataset('/g/data/qv56/replicas/input4MIPs/CMIP6/OMIP/MRI/MRI-JRA55-do-1-5-0/atmos/3hr/rsds/gr/v20200916/rsds_input4MIPs_atmosphericState_OMIP_MRI-JRA55-do-1-5-0_gr_201901010130-201912312230.nc')

In [ ]:
# Load a single point:
lon = -200.8
lat = -30.05
rsds_om2_hr = ds_hour.swflx.sel(xt_ocean=lon,yt_ocean=lat,method='nearest').sel(time=slice('2019-01-01','2019-01-02')).load()
rsds_om2_ts = ds_ts.swflx.sel(xt_ocean=lon,yt_ocean=lat,method='nearest').sel(time=slice('2019-01-01','2019-01-02')).load()
rsds_jra55 = ds_jra55.rsds.sel(lon=lon+360.,lat=lat,method='nearest').sel(time=slice('2019-01-01','2019-01-02')).load()

In [ ]:
# Plot a time series:
fig = plt.figure(figsize=(15,6))
rsds_om2_ts.plot(label='ACCESS-OM2 (shortwave in, swflx, time step)',marker='.',linewidth=2,markersize=10)
rsds_om2_hr.plot(label='ACCESS-OM2 (shortwave in, swflx, hourly)',marker='.',linewidth=2,markersize=10)
rsds_jra55.plot(label='JRA55 (shortwave down, rsds, 3-hourly)',marker='.',linewidth=2,markersize=10)
plt.grid()
plt.legend()
plt.title('Surface shortwave radiation at 30.05S, 159.2E in ACCESS-OM2-025 and JRA55')
plt.savefig('JRA55_solar_time_stepping.png',dpi=150,bbox_inches='tight')
#plt.set_xlim(['2019-01-01','2019-01-02'])